# Satellite Building Change Detection (LEVIR-CD)

CS57600 Machine Learning, Fall 2026 · Raviteja Gundu, Joel Adu-Kwarteng

Predict pixel-level building-change masks from pairs of satellite images, comparing an image-difference baseline, an early-fusion U-Net (U-Net-EF) and a Siamese U-Net.

**How to run**
1. Upload this notebook to Colab and choose **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Allow Drive access when prompted.

Data, checkpoints and results are stored on Google Drive under `CONFIG["project_root"]`, so they survive between Colab sessions. Each task adds one section to this notebook.

## Task 1 · Setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import PIL
import torch

# All settings live here. Later tasks add their own entries (data, baseline, models).
CONFIG = {
    "project_root": "/content/drive/MyDrive/levir-change-detection",  # persistent storage
    "local_root": "/content/local",  # fast VM disk, wiped when the session ends
    "seed": 42,
    "deterministic": False,  # True makes cuDNN deterministic (slower)
}

In [ ]:
def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed Python, NumPy and torch (CPU and all GPUs)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic


def seed_worker(worker_id: int) -> None:
    """DataLoader worker_init_fn: derive NumPy/Python seeds from torch's per-worker seed."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass(frozen=True)
class Paths:
    """Standard locations under the Drive project root, plus a fast local scratch folder."""

    root: Path
    local: Path

    @property
    def raw(self) -> Path:  # LEVIR-CD as downloaded: train/val/test/{A,B,label}
        return self.root / "data" / "raw"

    @property
    def tiles_zip(self) -> Path:  # 256x256 tiles (Task 3)
        return self.root / "data" / "tiles.zip"

    @property
    def stats_file(self) -> Path:  # training-set channel mean/std (Task 4)
        return self.root / "data" / "stats.json"

    @property
    def runs(self) -> Path:  # checkpoints and logs, one folder per run
        return self.root / "runs"

    @property
    def results(self) -> Path:  # metrics JSON and CSVs
        return self.root / "results"

    @property
    def figures(self) -> Path:  # figures for the report
        return self.results / "figures"

    @property
    def local_tiles(self) -> Path:  # tiles unzipped to local disk each session
        return self.local / "tiles"

    def run_dir(self, name: str) -> Path:
        return self.runs / name

    def make_dirs(self) -> "Paths":
        for p in (self.raw, self.runs, self.results, self.figures, self.local):
            p.mkdir(parents=True, exist_ok=True)
        return self

In [ ]:
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Google Drive is not mounted. Run the drive.mount cell above first.")

set_seed(CONFIG["seed"], CONFIG["deterministic"])
DEVICE = get_device()
PATHS = Paths(Path(CONFIG["project_root"]), Path(CONFIG["local_root"])).make_dirs()

print(f"torch {torch.__version__} | numpy {np.__version__} | pillow {PIL.__version__} | matplotlib {matplotlib.__version__}")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", DEVICE)
print()
for name in ("root", "raw", "tiles_zip", "stats_file", "runs", "results", "figures", "local_tiles"):
    print(f"{name:>11}: {getattr(PATHS, name)}")